In [3]:
%pip install -q -U langchain langchain-core langchain-google-genai "requests==2.32.4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.0 MB/s eta 0:00:00


 Imports & Setup

In [4]:
import os
import time
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage


def load_google_api_key() -> str:
    for key_name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
        value = os.getenv(key_name)
        if value:
            os.environ["GOOGLE_API_KEY"] = value
            os.environ["GEMINI_API_KEY"] = value
            return value
    try:
        from google.colab import userdata
        for key_name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
            try:
                value = userdata.get(key_name)
                if value:
                    os.environ["GOOGLE_API_KEY"] = value
                    os.environ["GEMINI_API_KEY"] = value
                    return value
            except Exception:
                pass
    except Exception:
        pass
    raise RuntimeError("Gemini API key not found.")


def message_to_text(message) -> str:
    text_attr = getattr(message, "text", None)
    if isinstance(text_attr, str) and text_attr.strip():
        return text_attr.strip()
    content = getattr(message, "content", None)
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                t = item.get("text")
                if t:
                    parts.append(t)
        return "\n".join(parts).strip()
    return ""


API_KEY = load_google_api_key()
MODEL_NAME = "gemini-2.5-flash"

llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0.3,
)

# Free tier allows 5 requests per minute — sleep between every LLM call
SLEEP_BETWEEN_CALLS = 15
SYSTEM_INSTRUCTION   = "You are a helpful and friendly AI assistant."

print(f"Loaded API key. Using model: {MODEL_NAME}")
print(f"Sleep between calls: {SLEEP_BETWEEN_CALLS}s (free tier rate limit protection)")

Loaded API key. Using model: gemini-2.5-flash
Sleep between calls: 15s (free tier rate limit protection)


Demonstrate Context Rot (The Problem)

In [6]:
def plain_chat(
    history: list,
    user_text: str,
    system_instruction: str,
) -> tuple:
    """Plain chat with NO memory management — history grows unbounded."""
    messages = [SystemMessage(content=system_instruction)]
    messages.extend(history)
    messages.append(HumanMessage(content=user_text))

    response = llm.invoke(messages)
    reply = message_to_text(response)
    time.sleep(SLEEP_BETWEEN_CALLS)

    updated_history = history + [
        HumanMessage(content=user_text),
        AIMessage(content=reply),
    ]
    return reply, updated_history


print("=" * 80)
print("DEMONSTRATING CONTEXT ROT — Plain chat with no memory management")
print("=" * 80)

plain_history = []

# These turns are designed so early facts (name, subject) are introduced first,
# then buried under many follow-up turns, then tested at the end.
context_rot_turns = [
    "My name is Arjun and I am studying Computer Science.",
    "I love working on AI projects and find NLP very interesting.",
    "What is the difference between supervised and unsupervised learning?",
    # This final question tests whether early context survived:
    "Do you remember what my name is and what subject I am studying?",
]

for turn in context_rot_turns:
    reply, plain_history = plain_chat(plain_history, turn, SYSTEM_INSTRUCTION)
    print(f"User     : {turn}")
    print(f"Assistant: {reply[:250]}")
    print(f"[History size: {len(plain_history)} messages]")
    print("-" * 80)

plain_final_answer = plain_history[-1].content
print(f"\nTotal raw messages in memory: {len(plain_history)}")
print("PROBLEM: As the conversation grows, early context gets diluted or lost.")

DEMONSTRATING CONTEXT ROT — Plain chat with no memory management
User     : My name is Arjun and I am studying Computer Science.
Assistant: Hi Arjun! It's great to meet you. Computer Science is a fantastic field, full of exciting possibilities.

Is there anything I can help you with regarding your studies, or anything else for that matter? I'm here to assist!
[History size: 2 messages]
--------------------------------------------------------------------------------
User     : I love working on AI projects and find NLP very interesting.
Assistant: That's fantastic, Arjun! You've picked two incredibly dynamic and impactful areas within Computer Science.

AI projects are always a thrill, and NLP is particularly fascinating because it sits at the intersection of human language and machine underst
[History size: 4 messages]
--------------------------------------------------------------------------------
User     : What is the difference between supervised and unsupervised learning?
Assistan

Hybrid Memory Implementation

In [7]:
# Hybrid Memory Config
WINDOW_SIZE = 4   # Number of recent raw messages to always keep


def summarize_history(old_messages: list, existing_summary: str) -> str:
    """Compress old messages into a running summary using the LLM."""

    conversation_lines = []
    for msg in old_messages:
        if isinstance(msg, HumanMessage):
            conversation_lines.append(f"User: {msg.content}")
        elif isinstance(msg, AIMessage):
            conversation_lines.append(f"Assistant: {msg.content}")

    conversation_block = "\n".join(conversation_lines)

    if existing_summary:
        prompt = f"""You are a memory manager for an AI assistant.

Existing summary of past conversation:
{existing_summary}

New conversation chunk to incorporate:
{conversation_block}

Write an updated summary that:
1. Preserves all important facts, names, preferences, and key decisions
2. Removes filler and redundant exchanges
3. Is concise and under 200 words

Updated summary:""".strip()

    else:
        prompt = f"""You are a memory manager for an AI assistant.

Conversation to summarize:
{conversation_block}

Write a concise summary that:
1. Preserves all important facts, names, preferences, and key decisions
2. Removes filler and redundant exchanges
3. Is under 200 words

Summary:""".strip()

    response = llm.invoke(prompt)
    time.sleep(SLEEP_BETWEEN_CALLS)
    return message_to_text(response)


def hybrid_chat(
    recent_history: list,
    running_summary: str,
    user_text: str,
    system_instruction: str,
) -> tuple:
    """
    Hybrid Memory Chat — solves context rot by combining:
    1. A compressed running summary of older messages
    2. A sliding window of the last WINDOW_SIZE raw messages

    Returns: (reply, updated_recent_history, updated_summary)
    """

    messages = []

    # Layer 1: System instruction
    messages.append(SystemMessage(content=system_instruction))

    # Layer 2: Inject running summary if it exists
    if running_summary:
        messages.append(SystemMessage(
            content=f"Summary of earlier conversation (for context):\n{running_summary}"
        ))

    # Layer 3: Recent raw messages (sliding window)
    messages.extend(recent_history)

    # Layer 4: Current user turn
    messages.append(HumanMessage(content=user_text))

    # Get LLM response
    response = llm.invoke(messages)
    reply = message_to_text(response)
    time.sleep(SLEEP_BETWEEN_CALLS)

    # Append new turn to recent history
    updated_recent = recent_history + [
        HumanMessage(content=user_text),
        AIMessage(content=reply),
    ]

    updated_summary = running_summary

    # If recent history exceeds window size, compress the overflow
    if len(updated_recent) > WINDOW_SIZE:
        old_messages   = updated_recent[:-WINDOW_SIZE]
        updated_recent = updated_recent[-WINDOW_SIZE:]

        print(f"  [Memory Manager] Compressing {len(old_messages)} messages into summary...")
        updated_summary = summarize_history(old_messages, running_summary)
        print(f"  [Memory Manager] Done. Window size: {len(updated_recent)} | Summary: {len(updated_summary)} chars")

    return reply, updated_recent, updated_summary


print("Hybrid Memory functions ready.")
print(f"Window size : last {WINDOW_SIZE} raw messages kept")
print("Older messages : compressed into a rolling summary")

Hybrid Memory functions ready.
Window size : last 4 raw messages kept
Older messages : compressed into a rolling summary


Run Hybrid Memory Demo

In [8]:
print("=" * 80)
print("HYBRID MEMORY DEMO — Same conversation, now with memory management")
print("=" * 80)

hybrid_recent_history = []
hybrid_running_summary = ""

# Exact same turns as the plain chat demo for fair comparison
hybrid_turns = [
    "My name is Arjun and I am studying Computer Science.",
    "I love working on AI projects and find NLP very interesting.",
    "What is the difference between supervised and unsupervised learning?",
    "Can you explain how transformers work in NLP?",
]

for turn in hybrid_turns:
    print(f"User     : {turn}")

    reply, hybrid_recent_history, hybrid_running_summary = hybrid_chat(
        recent_history=hybrid_recent_history,
        running_summary=hybrid_running_summary,
        user_text=turn,
        system_instruction=SYSTEM_INSTRUCTION,
    )

    print(f"Assistant: {reply[:300]}")
    print(f"[Window: {len(hybrid_recent_history)} msgs | Summary exists: {bool(hybrid_running_summary)}]")
    print("-" * 80)

hybrid_final_answer = hybrid_recent_history[-1].content

print("\nFinal Running Summary stored in memory:")
print("=" * 80)
print(hybrid_running_summary if hybrid_running_summary else "No summary generated yet.")

HYBRID MEMORY DEMO — Same conversation, now with memory management
User     : My name is Arjun and I am studying Computer Science.
Assistant: Hi Arjun! It's great to meet you. Computer Science is a fascinating field.

Is there anything I can help you with today, or anything you'd like to chat about related to your studies or anything else?
[Window: 2 msgs | Summary exists: False]
--------------------------------------------------------------------------------
User     : I love working on AI projects and find NLP very interesting.
Assistant: That's fantastic, Arjun! NLP (Natural Language Processing) is an incredibly dynamic and impactful area within AI. It's truly at the forefront of how humans and computers interact, and it's evolving at an astonishing pace.

What aspects of NLP do you find most interesting? Are you more into:

*   **L
[Window: 4 msgs | Summary exists: False]
--------------------------------------------------------------------------------
User     : What is the differe

Side-by-Side Comparison

In [9]:
print("=" * 80)
print("SIDE-BY-SIDE COMPARISON")
print("=" * 80)
print()
print("Test question asked at end of both conversations:")
print("'Do you remember what my name is and what subject I am studying?'")
print()

print("WITHOUT Hybrid Memory (Plain Chat):")
print("-" * 40)
print(plain_final_answer[:400])
print()

print("WITH Hybrid Memory (Hybrid Approach):")
print("-" * 40)
print(hybrid_final_answer[:400])
print()

print("=" * 80)
print("CONCLUSION")
print("=" * 80)
print(f"Plain chat stored  : {len(plain_history)} raw messages (grows unbounded)")
print(f"Hybrid chat stored : {len(hybrid_recent_history)} raw messages + compressed summary")
print()
print("Plain chat suffers from context rot — early facts get diluted over time.")
print("Hybrid memory solves this by:")
print("  1. Compressing old turns into a rolling summary (no token bloat)")
print("  2. Keeping recent turns raw (precise short-term recall)")
print("  3. Injecting both into every LLM call (complete context, minimal tokens)")

SIDE-BY-SIDE COMPARISON

Test question asked at end of both conversations:
'Do you remember what my name is and what subject I am studying?'

WITHOUT Hybrid Memory (Plain Chat):
----------------------------------------
Yes, I do!

Your name is **Arjun**, and you are studying **Computer Science**.

WITH Hybrid Memory (Hybrid Approach):
----------------------------------------
That's a fantastic follow-up question, Arjun! Transformers are truly the backbone of modern NLP, revolutionizing the field since their introduction in 2017 with the paper "Attention Is All You Need."

Before Transformers, Recurrent Neural Networks (RNNs) and their variants (LSTMs, GRUs) were dominant. While effective, they had limitations, especially with long sequences and parallel processing. Tr

CONCLUSION
Plain chat stored  : 8 raw messages (grows unbounded)
Hybrid chat stored : 4 raw messages + compressed summary

Plain chat suffers from context rot — early facts get diluted over time.
Hybrid memory solves thi